In [1]:
import nest
import os
import network
import addons
import numpy as np
import helpers
from network_params import net_dict
import network_params
from sim_params import sim_dict
from stimulus_params import stim_dict


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.7.0
 Built: Apr 15 2024 07:21:32

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



/home/hyc_1/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning:Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.


In [2]:
from scipy import signal as scipy_signal

def save_pop_activity(data_path, n_pops,
                      name="spike_recorder",
                      M=None, std=None,
                      num_neurons=None):
    """
    Replicates the pop_activity saving block from helpers.plot_raster
    without any plotting — safe for any number of populations.

    Parameters
    ----------
    data_path  : str, trial data directory (same as net.data_path)
    n_pops     : int, number of populations actually simulated
    name       : str, spike recorder label (default 'spike_recorder')
    M          : list of ints, Gaussian window widths per pop (default 50 each)
    std        : list of floats, Gaussian window stds per pop (default 10 each)
    num_neurons: list of ints, neuron counts per pop for normalisation;
                 if None, uses lesioned_net_dict['full_num_neurons']
    """
    if M   is None: M   = [50]  * n_pops
    if std is None: std = [10.0] * n_pops
    if num_neurons is None:
        num_neurons = list(lesioned_net_dict["full_num_neurons"])

    t0 = addons.analysis_dict["analysis_start"]
    t1 = addons.analysis_dict["analysis_end"]
    bin_size = addons.analysis_dict["convolve_bin_size"]

    sd_names, node_ids, data_analysis = helpers.__load_spike_times(
        data_path, name, t0, t1
    )

    for folder in ["measurements", "measurements/pop_activities",
                   "measurements/pop_activities2", "measurements/times"]:
        os.makedirs(os.path.join(data_path, folder), exist_ok=True)

    filtered = {}
    times_a  = {}
    for i, n in enumerate(sd_names):
        times_ = data_analysis[i]["time_ms"]
        pop_act, times_a[i] = np.histogram(times_, bins=int((t1 - t0) / bin_size))
        window     = scipy_signal.windows.gaussian(M[i], std[i])
        smoothed   = scipy_signal.convolve(pop_act.astype(float), window, mode="same")
        filtered[i] = smoothed

        np.savetxt(data_path + "measurements/pop_activities2/pop_activity_" + str(i) + ".dat",
                   smoothed / num_neurons[i])

        normed = (smoothed - smoothed.min()) / (smoothed.max() - smoothed.min() + 1e-12)
        np.savetxt(data_path + "measurements/pop_activities/pop_activity_"  + str(i) + ".dat",
                   normed)
        np.savetxt(data_path + "measurements/times/times_" + str(i) + ".dat",
                   times_a[i])

    return filtered, times_a

In [3]:
import copy
import numpy as np

# ── Helper: build a lesioned net_dict with chosen layers removed ──────────────

ALL_POPULATIONS = ["L23E", "L23I", "L4E", "L4I", "L5E", "L5I", "L6E", "L6I"]

def lesion_net_dict(base_dict, layers_to_remove):
    """
    Return a deep copy of net_dict with the specified layers removed.

    Parameters
    ----------
    base_dict       : the original net_dict
    layers_to_remove: list of str, e.g. ["L4E", "L4I"] or ["L5E", "L5I"]
                      Pass a whole layer name ("L4") to remove both E and I.

    Returns
    -------
    new_dict : modified net_dict (original is untouched)
    keep_idx : list of int — indices of surviving populations (useful for
               re-indexing spike recorder outputs afterwards)
    """
    # expand shorthand layer names  e.g. "L4" → ["L4E", "L4I"]
    expanded = []
    for name in layers_to_remove:
        if name in ALL_POPULATIONS:
            expanded.append(name)
        else:
            # treat as layer prefix: remove both E and I
            expanded += [p for p in ALL_POPULATIONS if p.startswith(name)]

    keep_idx = [i for i, p in enumerate(ALL_POPULATIONS) if p not in expanded]
    remove_idx = [i for i, p in enumerate(ALL_POPULATIONS) if p in expanded]

    if not remove_idx:
        raise ValueError(f"No matching populations found for: {layers_to_remove}")

    print(f"Removing populations : {[ALL_POPULATIONS[i] for i in remove_idx]}")
    print(f"Keeping  populations : {[ALL_POPULATIONS[i] for i in keep_idx]}")

    d = copy.deepcopy(base_dict)

    # 1-D arrays indexed by population
    for key in ("full_num_neurons", "full_mean_rates", "K_ext"):
        d[key] = d[key][keep_idx]

    # population name list
    d["populations"] = [ALL_POPULATIONS[i] for i in keep_idx]

    # 2-D arrays [target × source] — drop both rows and columns
    for key in ("conn_probs", "PSP_matrix_mean", "delay_matrix_mean"):
        mat = d[key]
        mat = mat[np.ix_(keep_idx, keep_idx)]
        d[key] = mat

    # thalamic connection probabilities (1-D, one entry per population)
    # only present after network_params updates stim_dict, so guard it
    # (stim_dict is separate, but conn_probs_th lives there — handled below)

    return d, keep_idx



In [4]:

# ── Configuration ─────────────────────────────────────────────────────────────

# Choose which layer(s) to remove.  Examples:
#   ["L4"]          → removes L4E + L4I
#   ["L5E", "L5I"]  → same effect written explicitly
#   ["L23", "L6"]   → removes both L23 and L6 (all four populations)


LAYERS_TO_REMOVE = ["L4"]   # ← edit this

lesioned_net_dict, keep_idx = lesion_net_dict(net_dict, LAYERS_TO_REMOVE)

# Adjust thalamic conn_probs if thalamic input is on
# (stim_dict["conn_probs_th"] is a 1-D array of length 8)
lesioned_stim_dict = copy.deepcopy(stim_dict)
if stim_dict.get("thalamic_input", False):
    lesioned_stim_dict["conn_probs_th"] = stim_dict["conn_probs_th"][keep_idx]

# ── Run lesioned simulations ──────────────────────────────────────────────────

bg_rate       = np.array([6, 9, 12, 18])
number_trials = 1
path          = "data_background_rate_fig_1_lesion3/"
label         = "_".join(LAYERS_TO_REMOVE) + "_removed"

for i in range(len(bg_rate)):
    run_path = path + str(round(bg_rate[i], 2)) + f"_{label}/"
    os.makedirs(run_path, exist_ok=True)

    for j in range(number_trials):
        nest.ResetKernel()

        net = network.Network(
            sim_dict,
            lesioned_net_dict,
            lesioned_stim_dict,
            path    = run_path + f"trial_{j}/",
            bg_rate = bg_rate[i],
        )

        net.create(rate="random")
        net.connect()

        net.simulate(sim_dict["t_presim"])
        net.simulate(sim_dict["t_sim"])

        if nest.Rank() == 0:
            pop_activity, times_a = save_pop_activity(
                data_path  = run_path + f"trial_{j}/",
                n_pops     = len(lesioned_net_dict["populations"]),
                num_neurons= list(lesioned_net_dict["full_num_neurons"]),
            )

Removing populations : ['L4E', 'L4I']
Keeping  populations : ['L23E', 'L23I', 'L5E', 'L5I', 'L6E', 'L6I']
Data will be written to: data_background_rate_fig_1_lesion3/6_L4_removed/trial_0/
  Directory has been created.


RNG seed: 55
Total number of virtual processes: 10
Creating neuronal populations.

Jun 29 12:31:50 SimulationManager::set_status [Info]: 
    Temporal resolution changed from 0.1 to 0.1 ms.
Creating recording devices.
  Creating spike recorders.
Creating Poisson generators for background input.
Connecting neuronal populations recurrently.
NodeCollection(metadata=None, model=iaf_psc_exp, size=20683, first=1, last=20683)
NodeCollection(metadata=None, model=iaf_psc_exp, size=5834, first=20684, last=26517)
NodeCollection(metadata=None, model=iaf_psc_exp, size=4850, first=26518, last=31367)
NodeCollection(metadata=None, model=iaf_psc_exp, size=1065, first=31368, last=32432)
NodeCollection(metadata=None, model=iaf_psc_exp, size=14395, first=32433, last=46827)
NodeCollection(m